# 14 — AIA + SHARP + GOES Fusion Dataset Alignment Protocol

**Purpose.** This notebook is a CPU-only alignment and audit notebook. It prepares the bridge from AIA-only image modelling to multimodal fusion.

It does **not** train any model.

## Scientific goal

Build a leakage-safe fusion-ready register that connects:

1. **AIA EUV image samples** — multi-wavelength active-region cutouts stored as `.npz`.
2. **HMI/SHARP magnetic parameters** — active-region magnetic field descriptors at 96-minute cadence.
3. **GOES flare/XRS information** — currently available as the AR-specific forecast label lineage; future GOES-history input features are audited separately.

## Important label rule

The official modelling label remains:

`label_48h_final`

This is the active-region-specific M/X flare label for the next 48 hours. Old embedded `.npz` labels must not be used.


In [ ]:
from pathlib import Path
import json
from datetime import datetime
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / ".git").exists() and (p / "training").exists():
            return p
    for p in candidates:
        if (p / "training").exists() and (p / "results").exists():
            return p
    raise RuntimeError("Could not find project root. Run from inside the solar-flare-aia-training repo.")

ROOT = find_project_root()
TRAINING_DIR = ROOT / "training"
METRICS_DIR = ROOT / "results" / "metrics"
FIG_DIR = ROOT / "results" / "figures"
FUSION_DIR = TRAINING_DIR / "fusion_manifests"
NOTEBOOK_DIR = ROOT / "notebooks" / "training"

for d in [METRICS_DIR, FIG_DIR, FUSION_DIR, NOTEBOOK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("TRAINING_DIR:", TRAINING_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("FUSION_DIR:", FUSION_DIR)
print("Started:", datetime.now().isoformat(timespec="seconds"))


## 1. Configuration and source files

In [ ]:
FILES = {
    "aia_baseline_manifest_2010_2016": TRAINING_DIR / "final_metadata" / "baseline_2010_2016_AR_SPECIFIC_manifest.csv",
    "sharp_curated_2010_2026": TRAINING_DIR / "final_metadata" / "curated_sharp_suryabench_true96min_48h_AR_SPECIFIC_2010_2026.csv",
    "aia_master_manifest_2010_2026": TRAINING_DIR / "manifests" / "aia_master_manifest_2010_2026.csv",
    "aia_master_manifest_2010_2026_labelled": TRAINING_DIR / "manifests" / "aia_master_manifest_2010_2026_LABELLED.csv",
}

LABEL_COL = "label_48h_final"

PREFERRED_SHARP_FEATURES_16 = [
    "MEANGBZ", "MEANGAM", "MEANGBT", "MEANGBH",
    "MEANJZD", "TOTUSJZ", "MEANALP", "MEANJZH",
    "ABSNJZH", "SAVNCPP", "MEANSHR", "SHRGT45",
    "R_VALUE", "USFLUX", "TOTPOT", "TOTUSJH",
]
OPTIONAL_SHARP_FEATURES = ["MEANPOT", "AREA_ACR"]
LOOKBACK_HOURS = [6, 12, 24, 48]
CADENCE_MINUTES = 96

FOLD_SPECS = {
    "fusion_test_2013": {"train_years": [2010, 2011], "val_years": [2012], "test_years": [2013]},
    "fusion_test_2014": {"train_years": [2010, 2011, 2012], "val_years": [2013], "test_years": [2014]},
    "fusion_test_2015": {"train_years": [2010, 2011, 2012, 2013], "val_years": [2014], "test_years": [2015]},
}

inventory_rows = []
for name, path in FILES.items():
    inventory_rows.append({
        "source_name": name,
        "path": str(path.relative_to(ROOT)),
        "exists": path.exists(),
        "size_mb": path.stat().st_size / (1024**2) if path.exists() else np.nan,
    })

inventory = pd.DataFrame(inventory_rows)
inventory_path = METRICS_DIR / "aia_sharp_goes_fusion_file_inventory.csv"
inventory.to_csv(inventory_path, index=False)

print("Saved:", inventory_path)
display(inventory)

missing = [str(p) for p in [FILES["aia_baseline_manifest_2010_2016"], FILES["sharp_curated_2010_2026"]] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required alignment input file(s):\n" + "\n".join(missing))


## 2. Load AIA and SHARP tables

In [ ]:
def load_csv(path, parse_dates=None):
    df = pd.read_csv(path)
    if parse_dates:
        for c in parse_dates:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

aia = load_csv(FILES["aia_baseline_manifest_2010_2016"], parse_dates=["T_REC_dt"])
sharp = load_csv(FILES["sharp_curated_2010_2026"], parse_dates=["T_REC_dt"])

aia_master = load_csv(FILES["aia_master_manifest_2010_2026"], parse_dates=["T_REC_dt"]) if FILES["aia_master_manifest_2010_2026"].exists() else None
aia_master_labelled = load_csv(FILES["aia_master_manifest_2010_2026_labelled"], parse_dates=["T_REC_dt"]) if FILES["aia_master_manifest_2010_2026_labelled"].exists() else None

for df in [aia, sharp, aia_master, aia_master_labelled]:
    if df is None:
        continue
    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
    if "HARPNUM" in df.columns:
        df["HARPNUM"] = pd.to_numeric(df["HARPNUM"], errors="coerce").astype("Int64")
    if "NOAA_AR_clean" in df.columns:
        df["NOAA_AR_clean"] = pd.to_numeric(df["NOAA_AR_clean"], errors="coerce").astype("Int64")
    if "NOAA_AR" in df.columns and "NOAA_AR_clean" not in df.columns:
        df["NOAA_AR_clean"] = pd.to_numeric(df["NOAA_AR"], errors="coerce").astype("Int64")
    if "year" not in df.columns and "T_REC_dt" in df.columns:
        df["year"] = df["T_REC_dt"].dt.year

print("AIA baseline:", aia.shape)
print("SHARP curated:", sharp.shape)
if aia_master is not None:
    print("AIA master:", aia_master.shape)
if aia_master_labelled is not None:
    print("AIA master labelled:", aia_master_labelled.shape)

display(aia.head(3))
display(sharp.head(3))


## 3. Column audit

In [ ]:
def column_audit_for_df(name, df):
    rows = []
    for c in df.columns:
        lower = c.lower()
        if c in ["sample_id", "T_REC_dt", "HARPNUM", "NOAA_AR", "NOAA_AR_clean"]:
            role = "alignment_key"
        elif c == LABEL_COL or "label" in lower or c == "y":
            role = "label_or_label_lineage"
        elif c in PREFERRED_SHARP_FEATURES_16:
            role = "preferred_sharp_feature"
        elif c in OPTIONAL_SHARP_FEATURES:
            role = "optional_sharp_feature"
        elif any(k in lower for k in ["goes", "xrs", "flare", "peak", "event", "class"]):
            role = "goes_or_event_candidate"
        elif any(k in lower for k in ["gcp", "path", "file", "s3", "nc", "npz"]):
            role = "file_path"
        else:
            role = "metadata"
        rows.append({
            "table_name": name,
            "column": c,
            "role_guess": role,
            "dtype": str(df[c].dtype),
            "non_null_preview_count": int(df[c].head(1000).notna().sum()),
        })
    return rows

audit_rows = []
audit_rows.extend(column_audit_for_df("aia_baseline_manifest_2010_2016", aia))
audit_rows.extend(column_audit_for_df("sharp_curated_2010_2026", sharp))
if aia_master is not None:
    audit_rows.extend(column_audit_for_df("aia_master_manifest_2010_2026", aia_master))
if aia_master_labelled is not None:
    audit_rows.extend(column_audit_for_df("aia_master_manifest_2010_2026_LABELLED", aia_master_labelled))

column_audit = pd.DataFrame(audit_rows)
column_audit_path = METRICS_DIR / "aia_sharp_goes_fusion_column_audit.csv"
column_audit.to_csv(column_audit_path, index=False)

print("Saved:", column_audit_path)
display(column_audit.groupby(["table_name", "role_guess"]).size().reset_index(name="n_columns"))


## 4. AIA ↔ SHARP alignment by sample_id

In [ ]:
aia_core_cols = [
    "sample_id", "gcp_path", "file", "T_REC_dt", "HARPNUM", "NOAA_AR_clean",
    "year", "label_48h_global_old", "label_48h_ar_specific", "label_48h_final",
    "label_48h", "y", "manifest_label_before_repair"
]
aia_core_cols = [c for c in aia_core_cols if c in aia.columns]
aia_core = aia[aia_core_cols].copy()

sharp_base_cols = [
    "sample_id", "T_REC", "T_REC_dt", "HARPNUM", "NOAA_AR_clean", "NOAA_ARS",
    "QUALITY", "year", "source_median_cadence_minutes", "target_cadence_minutes",
    "center_lon", "center_lat", "LON_CENTER",
    "label_48h_global_old", "label_48h_ar_specific", "label_48h_final", "label_48h",
]
sharp_feature_candidates = [f for f in PREFERRED_SHARP_FEATURES_16 + OPTIONAL_SHARP_FEATURES if f in sharp.columns]
sharp_cols = [c for c in sharp_base_cols + sharp_feature_candidates if c in sharp.columns]
sharp_core = sharp[sharp_cols].copy()
sharp_core = sharp_core.rename(columns={c: f"sharp_{c}" for c in sharp_core.columns if c != "sample_id"})

aligned = aia_core.merge(sharp_core, on="sample_id", how="left", indicator="sample_id_merge_status")

aligned["has_sharp_row"] = aligned["sample_id_merge_status"].eq("both")
aligned["has_aia_npz_path"] = aligned["gcp_path"].notna() if "gcp_path" in aligned.columns else False

if "T_REC_dt" in aligned.columns and "sharp_T_REC_dt" in aligned.columns:
    aligned["aia_sharp_time_delta_seconds"] = (
        pd.to_datetime(aligned["T_REC_dt"], errors="coerce")
        - pd.to_datetime(aligned["sharp_T_REC_dt"], errors="coerce")
    ).dt.total_seconds().abs()
else:
    aligned["aia_sharp_time_delta_seconds"] = np.nan

if "HARPNUM" in aligned.columns and "sharp_HARPNUM" in aligned.columns:
    aligned["harpnum_match"] = aligned["HARPNUM"].astype("Int64").eq(aligned["sharp_HARPNUM"].astype("Int64"))
else:
    aligned["harpnum_match"] = np.nan

if "NOAA_AR_clean" in aligned.columns and "sharp_NOAA_AR_clean" in aligned.columns:
    aligned["noaa_ar_match"] = aligned["NOAA_AR_clean"].astype("Int64").eq(aligned["sharp_NOAA_AR_clean"].astype("Int64"))
else:
    aligned["noaa_ar_match"] = np.nan

if LABEL_COL in aligned.columns and f"sharp_{LABEL_COL}" in aligned.columns:
    aligned["label_match_aia_vs_sharp"] = aligned[LABEL_COL].astype("float").eq(aligned[f"sharp_{LABEL_COL}"].astype("float"))
else:
    aligned["label_match_aia_vs_sharp"] = np.nan

aligned["official_label_48h_final"] = aligned[LABEL_COL]
aligned["goes_label_lineage_status"] = "goes_flare_catalog_used_for_ar_specific_future_label"
aligned["goes_input_feature_status"] = "not_yet_available_as_input_feature_table"

alignment_summary = pd.DataFrame([
    {"metric": "aia_rows", "value": len(aia_core)},
    {"metric": "sharp_rows", "value": len(sharp)},
    {"metric": "aligned_rows", "value": len(aligned)},
    {"metric": "rows_with_sharp_match", "value": int(aligned["has_sharp_row"].sum())},
    {"metric": "rows_without_sharp_match", "value": int((~aligned["has_sharp_row"]).sum())},
    {"metric": "sample_id_match_rate", "value": float(aligned["has_sharp_row"].mean())},
    {"metric": "harpnum_mismatch_count", "value": int((aligned["harpnum_match"] == False).sum()) if "harpnum_match" in aligned else np.nan},
    {"metric": "noaa_mismatch_count", "value": int((aligned["noaa_ar_match"] == False).sum()) if "noaa_ar_match" in aligned else np.nan},
    {"metric": "label_mismatch_count_aia_vs_sharp", "value": int((aligned["label_match_aia_vs_sharp"] == False).sum()) if "label_match_aia_vs_sharp" in aligned else np.nan},
    {"metric": "positive_samples_official_label", "value": int(aligned["official_label_48h_final"].fillna(0).astype(int).sum())},
    {"metric": "negative_samples_official_label", "value": int((aligned["official_label_48h_final"].fillna(0).astype(int) == 0).sum())},
])

alignment_summary_path = METRICS_DIR / "aia_sharp_goes_fusion_alignment_summary.csv"
alignment_summary.to_csv(alignment_summary_path, index=False)

alignment_manifest_path = FUSION_DIR / "aia_sharp_goes_fusion_alignment_manifest_2010_2016.csv"
aligned.to_csv(alignment_manifest_path, index=False)

print("Saved:", alignment_summary_path)
print("Saved:", alignment_manifest_path)
display(alignment_summary)
display(aligned.head(3))


## 5. SHARP feature availability audit

In [ ]:
feature_rows = []
for f in PREFERRED_SHARP_FEATURES_16:
    feature_rows.append({
        "feature": f,
        "feature_group": "preferred_16",
        "available_in_sharp_table": f in sharp.columns,
        "available_in_alignment_manifest": f"sharp_{f}" in aligned.columns,
        "missing_rate_in_aligned_rows": float(aligned[f"sharp_{f}"].isna().mean()) if f"sharp_{f}" in aligned.columns else np.nan,
    })

for f in OPTIONAL_SHARP_FEATURES:
    feature_rows.append({
        "feature": f,
        "feature_group": "optional_present_if_available",
        "available_in_sharp_table": f in sharp.columns,
        "available_in_alignment_manifest": f"sharp_{f}" in aligned.columns,
        "missing_rate_in_aligned_rows": float(aligned[f"sharp_{f}"].isna().mean()) if f"sharp_{f}" in aligned.columns else np.nan,
    })

sharp_feature_audit = pd.DataFrame(feature_rows)
sharp_feature_audit_path = METRICS_DIR / "aia_sharp_goes_fusion_sharp_feature_availability.csv"
sharp_feature_audit.to_csv(sharp_feature_audit_path, index=False)

available_preferred = sharp_feature_audit[
    (sharp_feature_audit["feature_group"] == "preferred_16") &
    (sharp_feature_audit["available_in_sharp_table"])
]["feature"].tolist()
missing_preferred = sharp_feature_audit[
    (sharp_feature_audit["feature_group"] == "preferred_16") &
    (~sharp_feature_audit["available_in_sharp_table"])
]["feature"].tolist()

print("Saved:", sharp_feature_audit_path)
print("Available preferred SHARP features:", available_preferred)
print("Missing preferred SHARP features:", missing_preferred)
display(sharp_feature_audit)


## 6. SHARP temporal sequence availability audit

In [ ]:
sharp_time = sharp[["HARPNUM", "T_REC_dt", "sample_id"]].dropna(subset=["HARPNUM", "T_REC_dt"]).copy()
sharp_time["HARPNUM"] = sharp_time["HARPNUM"].astype(int)
sharp_time["t_ns"] = pd.to_datetime(sharp_time["T_REC_dt"]).astype("int64")
sharp_time = sharp_time.sort_values(["HARPNUM", "t_ns"])

harp_to_times = {
    int(h): sub["t_ns"].to_numpy()
    for h, sub in sharp_time.groupby("HARPNUM")
}

seq_df = aligned[["sample_id", "T_REC_dt", "HARPNUM", "year", "official_label_48h_final", "has_sharp_row"]].copy()
seq_df["T_REC_dt"] = pd.to_datetime(seq_df["T_REC_dt"], errors="coerce")
seq_df["HARPNUM_int"] = pd.to_numeric(seq_df["HARPNUM"], errors="coerce")

for hours in LOOKBACK_HOURS:
    col = f"sharp_history_count_{hours}h"
    ok_col = f"has_min_sharp_history_{hours}h"
    expected_steps = int(np.floor((hours * 60) / CADENCE_MINUTES)) + 1

    counts = []
    for _, r in seq_df.iterrows():
        if pd.isna(r["HARPNUM_int"]) or pd.isna(r["T_REC_dt"]):
            counts.append(0)
            continue
        times = harp_to_times.get(int(r["HARPNUM_int"]))
        if times is None:
            counts.append(0)
            continue
        t = pd.Timestamp(r["T_REC_dt"]).value
        start = t - int(hours * 3600 * 1_000_000_000)
        left = np.searchsorted(times, start, side="left")
        right = np.searchsorted(times, t, side="right")
        counts.append(int(max(0, right - left)))

    seq_df[col] = counts
    seq_df[ok_col] = seq_df[col] >= expected_steps
    seq_df[f"expected_steps_{hours}h"] = expected_steps

seq_availability_path = METRICS_DIR / "aia_sharp_goes_fusion_sharp_sequence_availability_per_sample.csv"
seq_df.to_csv(seq_availability_path, index=False)

summary_rows = []
for hours in LOOKBACK_HOURS:
    count_col = f"sharp_history_count_{hours}h"
    ok_col = f"has_min_sharp_history_{hours}h"
    summary_rows.append({
        "lookback_hours": hours,
        "expected_steps": int(seq_df[f"expected_steps_{hours}h"].iloc[0]),
        "rows": len(seq_df),
        "mean_history_count": float(seq_df[count_col].mean()),
        "median_history_count": float(seq_df[count_col].median()),
        "rows_meeting_min_history": int(seq_df[ok_col].sum()),
        "coverage_rate": float(seq_df[ok_col].mean()),
        "positive_coverage_rate": float(seq_df.loc[seq_df["official_label_48h_final"] == 1, ok_col].mean()) if (seq_df["official_label_48h_final"] == 1).any() else np.nan,
        "negative_coverage_rate": float(seq_df.loc[seq_df["official_label_48h_final"] == 0, ok_col].mean()) if (seq_df["official_label_48h_final"] == 0).any() else np.nan,
    })

seq_summary = pd.DataFrame(summary_rows)
seq_summary_path = METRICS_DIR / "aia_sharp_goes_fusion_sharp_sequence_availability_summary.csv"
seq_summary.to_csv(seq_summary_path, index=False)

print("Saved:", seq_availability_path)
print("Saved:", seq_summary_path)
display(seq_summary)


## 7. GOES label-lineage and input-feature readiness audit

In [ ]:
GOES_EXPLICIT_KEYWORDS = ["goes", "xrs", "soft_xray", "xrsa", "xrsb", "xray", "1_8", "0_5_4"]
EVENT_HISTORY_KEYWORDS = ["flare", "peak", "event", "class", "background"]
LABEL_KEYWORDS = ["label_48h", "label", "y"]

scan_rows = []
for p in ROOT.rglob("*.csv"):
    try:
        df0 = pd.read_csv(p, nrows=1)
    except Exception:
        continue

    cols = list(df0.columns)
    lower_cols = {c: c.lower() for c in cols}
    explicit_goes_cols = [c for c, lc in lower_cols.items() if any(k in lc for k in GOES_EXPLICIT_KEYWORDS)]
    event_cols = [c for c, lc in lower_cols.items() if any(k in lc for k in EVENT_HISTORY_KEYWORDS)]
    label_cols = [c for c, lc in lower_cols.items() if any(k in lc for k in LABEL_KEYWORDS)]

    if explicit_goes_cols or event_cols or label_cols:
        scan_rows.append({
            "path": str(p.relative_to(ROOT)),
            "size_mb": p.stat().st_size / (1024**2),
            "explicit_goes_or_xrs_columns": ";".join(explicit_goes_cols),
            "event_or_flare_candidate_columns": ";".join(event_cols),
            "label_columns": ";".join(label_cols),
        })

goes_scan = pd.DataFrame(scan_rows)
goes_scan_path = METRICS_DIR / "aia_sharp_goes_fusion_goes_column_scan.csv"
goes_scan.to_csv(goes_scan_path, index=False)

has_explicit_goes_inputs = False
if not goes_scan.empty:
    has_explicit_goes_inputs = goes_scan["explicit_goes_or_xrs_columns"].fillna("").str.len().gt(0).any()

goes_readiness = pd.DataFrame([
    {
        "item": "goes_future_flare_label_lineage",
        "status": "available_via_label_48h_final_and_ar_specific_label_columns",
        "leakage_risk": "safe_if_used_only_as_target_label",
        "next_action": "preserve label_48h_final as official target",
    },
    {
        "item": "goes_past_xrs_input_features",
        "status": "available" if has_explicit_goes_inputs else "not_found_in_current_repository_scan",
        "leakage_risk": "must_use_only_times_before_sample_T_REC_dt",
        "next_action": "extract GOES history features in a separate notebook if needed",
    },
    {
        "item": "recent_flare_history_features",
        "status": "not_confirmed_as_clean_input_feature_table",
        "leakage_risk": "future flare events inside 48h forecast window must never be inputs",
        "next_action": "design past-only features: past_6h_12h_24h flare counts, time_since_last_MX, pre-issue background flux",
    },
])

goes_readiness_path = METRICS_DIR / "aia_sharp_goes_fusion_goes_readiness_audit.csv"
goes_readiness.to_csv(goes_readiness_path, index=False)

print("Saved:", goes_scan_path)
print("Saved:", goes_readiness_path)
print("Explicit GOES/XRS input columns found:", has_explicit_goes_inputs)
display(goes_readiness)
display(goes_scan.head(20))


## 8. Chronological fusion fold assignments

In [ ]:
fold_rows = []
for fold_id, spec in FOLD_SPECS.items():
    for split_name, years in [
        ("train", spec["train_years"]),
        ("val", spec["val_years"]),
        ("test", spec["test_years"]),
    ]:
        sub = aligned[aligned["year"].isin(years)].copy()
        sub["fusion_fold_id"] = fold_id
        sub["fusion_split"] = split_name
        sub["fusion_split_years"] = ",".join(map(str, years))
        fold_rows.append(sub)

fold_assignments = pd.concat(fold_rows, ignore_index=True) if fold_rows else pd.DataFrame()
fold_assignments_path = FUSION_DIR / "aia_sharp_goes_fusion_fold_assignments_2013_2015.csv"
fold_assignments.to_csv(fold_assignments_path, index=False)

fold_summary = (
    fold_assignments
    .groupby(["fusion_fold_id", "fusion_split"], as_index=False)
    .agg(
        rows=("sample_id", "count"),
        positives=("official_label_48h_final", "sum"),
        sharp_matched=("has_sharp_row", "sum"),
        mean_year=("year", "mean"),
    )
)
fold_summary["negatives"] = fold_summary["rows"] - fold_summary["positives"]
fold_summary["positive_rate"] = fold_summary["positives"] / fold_summary["rows"]
fold_summary["sharp_match_rate"] = fold_summary["sharp_matched"] / fold_summary["rows"]

fold_summary_path = METRICS_DIR / "aia_sharp_goes_fusion_fold_summary.csv"
fold_summary.to_csv(fold_summary_path, index=False)

print("Saved:", fold_assignments_path)
print("Saved:", fold_summary_path)
display(fold_summary)


## 9. Visual checks

In [ ]:
import matplotlib.pyplot as plt

def save_bar(df, x, y, title, filename, rotation=30):
    fig = plt.figure(figsize=(10, 5))
    plt.bar(df[x].astype(str), df[y])
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

year_summary = (
    aligned.groupby("year", as_index=False)
    .agg(
        rows=("sample_id", "count"),
        positives=("official_label_48h_final", "sum"),
        sharp_matched=("has_sharp_row", "sum"),
    )
)
year_summary["positive_rate"] = year_summary["positives"] / year_summary["rows"]
year_summary["sharp_match_rate"] = year_summary["sharp_matched"] / year_summary["rows"]

save_bar(year_summary, "year", "rows", "Fusion candidate rows by year", "aia_sharp_goes_fusion_rows_by_year.png")
save_bar(year_summary, "year", "positive_rate", "Official label positive rate by year", "aia_sharp_goes_fusion_positive_rate_by_year.png")
save_bar(year_summary, "year", "sharp_match_rate", "AIA-SHARP sample_id match rate by year", "aia_sharp_goes_fusion_sharp_match_rate_by_year.png")

seq_plot = seq_summary.copy()
seq_plot["lookback"] = seq_plot["lookback_hours"].astype(str) + "h"
save_bar(seq_plot, "lookback", "coverage_rate", "SHARP sequence coverage by lookback window", "aia_sharp_goes_fusion_sharp_sequence_coverage.png")


## 10. Write fusion protocol and research-log update

In [ ]:
def fmt_pct(x):
    try:
        if pd.isna(x):
            return "NA"
        return f"{100*float(x):.2f}%"
    except Exception:
        return str(x)

match_rate = float(aligned["has_sharp_row"].mean())
label_mismatch_count = int((aligned["label_match_aia_vs_sharp"] == False).sum()) if "label_match_aia_vs_sharp" in aligned else -1
available_features = [f for f in PREFERRED_SHARP_FEATURES_16 if f in sharp.columns]
missing_features = [f for f in PREFERRED_SHARP_FEATURES_16 if f not in sharp.columns]
best_lookback = seq_summary.sort_values("lookback_hours").iloc[-1]
goes_input_status = "available" if has_explicit_goes_inputs else "not found as explicit GOES/XRS input columns in this repository scan"

protocol_text = f"""
# AIA + SHARP + GOES Fusion Alignment Protocol

**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Purpose

This protocol prepares the next multimodal flare-forecasting stage. It aligns AIA image samples with SHARP magnetic metadata and audits the current status of GOES information.

## Source files

- AIA baseline modelling manifest: `training/final_metadata/baseline_2010_2016_AR_SPECIFIC_manifest.csv`
- SHARP curated metadata table: `training/final_metadata/curated_sharp_suryabench_true96min_48h_AR_SPECIFIC_2010_2026.csv`
- AIA master manifest: `training/manifests/aia_master_manifest_2010_2026.csv`
- AIA master labelled manifest: `training/manifests/aia_master_manifest_2010_2026_LABELLED.csv`

## Official label

The official target label remains `label_48h_final`.

This label represents the active-region-specific future M/X flare label within the 48-hour forecast window. Embedded `.npz` labels are not used as scientific truth.

## AIA-SHARP alignment

Primary alignment key: `sample_id`.

Secondary checks:

- `HARPNUM`
- `NOAA_AR_clean`
- `T_REC_dt`
- label consistency between AIA and SHARP tables

AIA-SHARP sample match rate: **{fmt_pct(match_rate)}**.

AIA-vs-SHARP label mismatch count: **{label_mismatch_count}**.

## SHARP feature audit

Preferred SHARP features available:

`{available_features}`

Preferred SHARP features missing:

`{missing_features}`

Missing features must not be silently imputed into the feature set.

## SHARP temporal sequence design

Candidate lookback windows audited:

`{LOOKBACK_HOURS}` hours

The long-lookback audit row is:

- lookback_hours: {best_lookback['lookback_hours']}
- expected_steps: {best_lookback['expected_steps']}
- coverage_rate: {fmt_pct(best_lookback['coverage_rate'])}

Recommended first fusion temporal branch:

- start with **24h SHARP history** if coverage is strong;
- use 48h only if coverage remains acceptable and computation is manageable;
- preserve train-only normalisation statistics.

## GOES status

GOES is currently present as the source lineage behind the flare labels, through `label_48h_final` and the AR-specific label columns.

GOES input-feature status: **{goes_input_status}**.

Therefore, the next stage should treat GOES in two layers:

1. **Target supervision:** already active through `label_48h_final`.
2. **Optional input branch:** requires a separate past-only GOES/XRS feature extraction notebook.

Safe GOES input features must use only information before the issue time `T_REC_dt`.

Unsafe GOES leakage:

- any flare or XRS information from the 48h forecast window;
- any peak class/event information after the issue timestamp.

## Fold protocol

Fusion folds generated:

- `fusion_test_2013`
- `fusion_test_2014`
- `fusion_test_2015`

## Outputs

- `training/fusion_manifests/aia_sharp_goes_fusion_alignment_manifest_2010_2016.csv`
- `training/fusion_manifests/aia_sharp_goes_fusion_fold_assignments_2013_2015.csv`
- `results/metrics/aia_sharp_goes_fusion_*.csv`
- `results/metrics/aia_sharp_goes_fusion_protocol.md`
- `results/figures/aia_sharp_goes_fusion_*.png`

## Next notebook

Recommended next notebook:

`15_sharp_temporal_baseline_from_fusion_manifest.ipynb`

Purpose:

Train a SHARP-only temporal baseline using the fusion manifest before introducing full AIA+SHARP(+GOES) neural fusion.
""".strip()

protocol_path = METRICS_DIR / "aia_sharp_goes_fusion_protocol.md"
protocol_path.write_text(protocol_text, encoding="utf-8")

research_log_update_path = METRICS_DIR / "aia_sharp_goes_fusion_research_log_update.md"
research_log_update_path.write_text(protocol_text, encoding="utf-8")

print(protocol_text)
print("\nSaved:", protocol_path)
print("Saved:", research_log_update_path)


## 11. Final inventory

In [ ]:
print("Generated fusion manifests:")
for p in sorted(FUSION_DIR.glob("aia_sharp_goes_fusion_*")):
    print(" -", p.relative_to(ROOT), f"({p.stat().st_size/1024:.1f} KiB)")

print("\nGenerated metrics/protocol files:")
for p in sorted(METRICS_DIR.glob("aia_sharp_goes_fusion_*")):
    print(" -", p.relative_to(ROOT), f"({p.stat().st_size/1024:.1f} KiB)")

print("\nGenerated figures:")
for p in sorted(FIG_DIR.glob("aia_sharp_goes_fusion_*.png")):
    print(" -", p.relative_to(ROOT), f"({p.stat().st_size/1024:.1f} KiB)")

print("\nFinished:", datetime.now().isoformat(timespec="seconds"))
